# 14 — SCC decision-boundary check after real MILK training
**CPU only.** No retraining, image downloads or variables from notebook 13 are needed. Read saved validation scores for the completed ResNet50 run `exp08_572b2a3e5ca2`. Check whether a bounded adjustment of class decision weights recovers SCC sensitivity while preserving the other classes.

Evaluate 27 predeclared policies on both validation sets separately. BCC weight is 1, melanoma weight is one of 0.75/1/1.25, and SCC weight is one of 1/1.25/1.5/2/3/4/6/8/12. Classification is `argmax(original_scores * weights)`. Weighted scores are **decision values**, not calibrated probabilities. Never relabel them as cancer risk.

A policy passes the **validation screen** only if **both** validation sets have accuracy ≥0.85, macro F1 >0.80, and each class recall ≥0.80. Passing this screen is not evidence of meeting test requirements. If no policy passes, preserve the baseline policy; report trade-offs rather than claiming a fix. No test files, test scores, model replacement or app changes are made. This bounded search cannot prove that every possible decision rule fails.

Skin-tone reliability cannot be established by adjusting a decision boundary; real subgroup evidence is still required.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT=Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
RUN_ID='exp08_572b2a3e5ca2'
MODEL_ROOT=ROOT/'Models'/'Transfer_Learning'/RUN_ID/'ResNet50'
REPORT_ROOT=ROOT/'Reports'/'Training'/RUN_ID/'ResNet50'
STAMP=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT_ROOT=ROOT/'Reports'/'SCC_Decision_Check'/RUN_ID/STAMP
OUTPUT_ROOT.mkdir(parents=True,exist_ok=False)
CLASS_NAMES=['BCC','Melanoma','SCC']

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()

def decision_metrics(labels,predicted):
    labels=np.asarray(labels,dtype=np.int64);predicted=np.asarray(predicted,dtype=np.int64)
    if labels.shape!=predicted.shape or labels.ndim!=1 or not len(labels):raise ValueError('Invalid label shapes.')
    if not np.isin(labels,[0,1,2]).all() or not np.isin(predicted,[0,1,2]).all():raise ValueError('Invalid class index.')
    matrix=np.zeros((3,3),dtype=np.int64)
    np.add.at(matrix,(labels,predicted),1)
    support=matrix.sum(axis=1)
    if (support==0).any():raise ValueError('All three classes must be present.')
    tp=matrix.diagonal().astype(float)
    precision=np.divide(tp,matrix.sum(axis=0),out=np.zeros(3),where=matrix.sum(axis=0)>0)
    recall=tp/support
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(3),where=(precision+recall)>0)
    metrics={'accuracy':float(tp.sum()/len(labels)),'macro_f1':float(f1.mean()),
             'BCC_recall':float(recall[0]),'Melanoma_recall':float(recall[1]),
             'SCC_precision':float(precision[2]),'SCC_recall':float(recall[2]),'SCC_f1':float(f1[2])}
    table=pd.DataFrame({'class_label':CLASS_NAMES,'precision':precision,'recall':recall,'f1':f1,'support':support,'true_positive':tp.astype(int),'false_negative':support-tp.astype(int)})
    return metrics,table,matrix

def targets_met(metrics):
    return bool(metrics['accuracy']>=.85 and metrics['macro_f1']>.80 and
                min(metrics['BCC_recall'],metrics['Melanoma_recall'],metrics['SCC_recall'])>=.80)

def apply_policy(probabilities,melanoma_weight,scc_weight):
    values=np.asarray(probabilities,dtype=float)
    if values.ndim!=2 or values.shape[1]!=3 or not np.isfinite(values).all():raise ValueError('Invalid score matrix.')
    if (values<0).any() or (values>1).any() or not np.allclose(values.sum(axis=1),1,atol=1e-4):raise ValueError('Expected original three-class softmax scores.')
    weights=np.array([1.,melanoma_weight,scc_weight],dtype=float)
    if not np.isfinite(weights).all() or (weights<=0).any():raise ValueError('Decision weights must be finite and positive.')
    return (values*weights).argmax(axis=1)

metadata=json.loads((MODEL_ROOT/'metadata.json').read_text())
if not metadata.get('training_complete') or metadata['run_id']!=RUN_ID:raise ValueError('Expected completed notebook 13 run.')
if metadata['class_names']!=CLASS_NAMES:raise ValueError('Class order mismatch.')
if sha256_file(MODEL_ROOT/'best.keras')!=metadata['model_sha256']:raise ValueError('Model checksum mismatch.')
MILK_ROOT=ROOT/'Datasets'/'MILK10k'/'Prepared'/'20261104_175741_606266'
validation_paths={'original':ROOT/'Datasets'/'Splits'/'validation_metadata.csv',
                  'MILK10k':MILK_ROOT/'validation_metadata.csv'}
expected_hashes={'original':metadata['experiment_settings']['original_split_sha256']['validation'],
                 'MILK10k':metadata['experiment_settings']['milk_split_sha256']['validation']}
frames={};scores={};labels={};provenance={}
for name,path in validation_paths.items():
    if sha256_file(path)!=expected_hashes[name]:raise ValueError('Validation manifest changed: '+name)
    expected=pd.read_csv(path)
    prediction_path=REPORT_ROOT/f'best_{name}_validation_predictions.csv'
    frame=pd.read_csv(prediction_path)
    required={'dataset','image_id','class_label','label_index','split',*[f'score_{c}' for c in CLASS_NAMES]}
    if not required.issubset(frame.columns):raise ValueError('Prediction columns missing: '+name)
    if len(frame)!=len(expected) or not frame.split.eq('validation').all():raise ValueError('Wrong validation predictions: '+name)
    if frame.duplicated(['dataset','image_id']).any():raise ValueError('Duplicate prediction IDs: '+name)
    merged=frame.merge(expected[['dataset','image_id','class_label','label_index']],on=['dataset','image_id'],how='outer',suffixes=('','_expected'),indicator=True,validate='one_to_one')
    if not merged['_merge'].eq('both').all() or not merged.class_label.eq(merged.class_label_expected).all() or not merged.label_index.eq(merged.label_index_expected).all():
        raise ValueError('Prediction identities or labels differ from validation manifest: '+name)
    if not frame.class_label.map({c:i for i,c in enumerate(CLASS_NAMES)}).eq(frame.label_index).all():raise ValueError('Label order changed: '+name)
    values=frame[[f'score_{c}' for c in CLASS_NAMES]].to_numpy(dtype=float)
    y=frame.label_index.to_numpy(dtype=np.int64)
    baseline,_,_=decision_metrics(y,apply_policy(values,1,1))
    reference=metadata['validation_results_by_source'][name]
    for key in ['accuracy','macro_f1','SCC_precision','SCC_recall','SCC_f1']:
        np.testing.assert_allclose(baseline[key],reference[key],rtol=0,atol=1e-7)
    frames[name]=frame;scores[name]=values;labels[name]=y
    provenance[name]={'validation_sha256':sha256_file(path),'prediction_sha256':sha256_file(prediction_path),'images':len(frame)}
    print(name,'saved validation reference: passed;',len(frame),'images')
print('No test files or images loaded. No GPU required.')

## Check the fixed decision grid
Original scores and labels stay unchanged. Report both datasets separately. A policy's robust macro F1 is the **minimum** of its two validation macro F1 scores; robust sensitivity is the minimum recall across all six dataset/class combinations. Policies that improve SCC recall at the cost of BCC/melanoma recall do not pass the screen.

In [ ]:
MELANOMA_WEIGHTS=[.75,1.,1.25]
SCC_WEIGHTS=[1.,1.25,1.5,2.,3.,4.,6.,8.,12.]
rows=[];policy_rows=[]
for mw in MELANOMA_WEIGHTS:
    for sw in SCC_WEIGHTS:
        policy_id=f'mel_{mw:g}_scc_{sw:g}'
        results={}
        for name in frames:
            metrics,_,_=decision_metrics(labels[name],apply_policy(scores[name],mw,sw))
            results[name]=metrics
            rows.append({'policy':policy_id,'validation_set':name,'melanoma_weight':mw,'scc_weight':sw,**metrics,'validation_targets_met':targets_met(metrics)})
        policy_rows.append({'policy':policy_id,'melanoma_weight':mw,'scc_weight':sw,
            'minimum_macro_f1':min(v['macro_f1'] for v in results.values()),
            'minimum_accuracy':min(v['accuracy'] for v in results.values()),
            'minimum_SCC_recall':min(v['SCC_recall'] for v in results.values()),
            'minimum_other_class_recall':min(min(v['BCC_recall'],v['Melanoma_recall']) for v in results.values()),
            'both_validation_sets_pass':all(targets_met(v) for v in results.values())})
results=pd.DataFrame(rows)
ranking=pd.DataFrame(policy_rows).sort_values(['both_validation_sets_pass','minimum_macro_f1','minimum_SCC_recall','scc_weight'],ascending=[False,False,False,True]).reset_index(drop=True)
results.to_csv(OUTPUT_ROOT/'all_validation_results.csv',index=False)
ranking.to_csv(OUTPUT_ROOT/'policy_ranking.csv',index=False)
print('Baseline, unchanged decision rule:')
display(results.loc[results.policy.eq('mel_1_scc_1')])
print('Top configurations (not test results):')
display(ranking.head(10))
passing=ranking.loc[ranking.both_validation_sets_pass]
if passing.empty:
    print('NO CONFIGURATION IN THIS GRID MET ALL VALIDATION TARGETS.')
    print('Baseline decision rule retained. No deployment policy exported.')
    chosen=ranking.iloc[0]
    tag='exploratory_only'
else:
    chosen=passing.iloc[0]
    tag='validation_candidate'
    print('A VALIDATION CANDIDATE WAS FOUND. Test targets remain unverified.')
print('Details for:',chosen['policy'])
display(results.loc[results.policy.eq(chosen['policy'])])
for name in frames:
    predicted=apply_policy(scores[name],float(chosen['melanoma_weight']),float(chosen['scc_weight']))
    metrics,table,matrix=decision_metrics(labels[name],predicted)
    print(name,'per-class results:')
    display(table)
    table.to_csv(OUTPUT_ROOT/f'{tag}_{name}_per_class.csv',index=False)
    pd.DataFrame(matrix,index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(OUTPUT_ROOT/f'{tag}_{name}_confusion_matrix.csv')
    out=frames[name].copy()
    out['decision_class']=[CLASS_NAMES[i] for i in predicted]
    out['decision_policy']=chosen['policy']
    out.to_csv(OUTPUT_ROOT/f'{tag}_{name}_predictions.csv',index=False)
# Show SCC recall >= 0.80 trade-offs even when no policy passes all criteria.
recall_options=ranking.loc[ranking.minimum_SCC_recall.ge(.80)]
print('Policies with SCC recall >= 0.80 on BOTH validation sets:',len(recall_options))
if len(recall_options):display(recall_options.head(5))
policy={'run_id':RUN_ID,'model_sha256':metadata['model_sha256'],'class_names':CLASS_NAMES,
        'operation':'argmax(original_scores * decision_weights)','probability_calibration':False,
        'grid':{'melanoma':MELANOMA_WEIGHTS,'SCC':SCC_WEIGHTS},'validation_sources':provenance,
        'validation_screen_passed':bool(len(passing)),'test_targets_verified':False,
        'app_model_replaced':False,'darker_skin_reliability_established':False,
        'decision_weights':[1.,float(chosen['melanoma_weight']),float(chosen['scc_weight'])] if len(passing) else [1.,1.,1.]}
(OUTPUT_ROOT/'decision_check_summary.json').write_text(json.dumps(policy,indent=2),encoding='utf-8')
if len(passing):
    (OUTPUT_ROOT/'candidate_decision_policy_REQUIRES_TEST_EVALUATION.json').write_text(json.dumps(policy,indent=2),encoding='utf-8')
print('SCC DECISION CHECK COMPLETE')
print('Validation screen passed:',bool(len(passing)))
print('Reports:',OUTPUT_ROOT)
print('No retraining, test evaluation or app changes performed.')

## Save the result
Save this executed notebook to Drive and GitHub as `notebooks/14_scc_decision_boundary_check.ipynb`.
Commit message: `Record SCC sensitivity and decision-boundary validation results`.

Share the final status and the detailed original/MILK results. If no policy passes, the current scores do not support an accepted decision rule **within this grid**: further work must improve discrimination rather than simply declaring SCC more often. A targeted SCC-versus-BCC specialist is a different model experiment; it must be judged on these same fixed real validation partitions. Do not change original labels, discard difficult held-out examples, or regenerate splits to meet targets.

If a policy passes, freeze that model checksum and decision policy before test evaluation. The original test set has already been inspected in earlier experiments; do not describe it as a fresh untouched final test. MILK prospective test results remain separate from original test results. No validation policy establishes darker-skin reliability.